# Notebook 05 — SqueezeNet 1.1 on ImageNet (pretrained)

Uses a **pretrained SqueezeNet 1.1** (1.24 M params, 58.2 % Top-1, no residual connections,
no FC layers) for BFT analysis. Weights download automatically from torchvision (~4.7 MB).

**Why SqueezeNet?**  
- Smallest pretrained ImageNet model with no residual connections  
- No large FC layers — the classifier is a single `Conv2d(512, 1000, 1)`, whose
  joint arbor matrix is only ~2 GB vs 75+ GB for AlexNet's first FC layer  

**Architecture note — squeeze spine:**  
SqueezeNet Fire modules have *parallel* expand branches (`expand1x1` + `expand3x3`).
Capturing all Conv2d layers would violate BFT's sequential assumption.
Instead, `collect_layer_dicts` is called with `layer_filter=squeezenet_spine_filter`,
which selects only the **squeeze-spine**: initial conv → 8 squeeze convs → classifier conv.
Each squeeze conv's `input_fmap` is the full concatenated output of the preceding Fire
module's expand branches — the true activation flowing through the network — so the
sequential chain is valid.

**Dataset:** Only the **validation set** is required (`ILSVRC2012_img_val.tar`, ~6.3 GB).
Set `IMAGENET_DIR` in §0 to a directory that contains a `val/` subfolder with the
50 000 validation images in ImageFolder layout (`val/<synset>/*.JPEG`).
No training images are needed.

## §0 — Imports & Config

In [1]:
%matplotlib inline
import sys, os, pickle, copy, warnings
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torchvision.models import squeezenet1_1, SqueezeNet1_1_Weights
from sklearn.metrics.pairwise import paired_cosine_distances
from torch.utils.data import DataLoader, Subset, TensorDataset

from src import (
    load_experiment, save_experiment,
    collect_layer_dicts,
    bft,
    build_scaffold_edges, scaffold_loading_from_edges, plot_scaffold_graph,
    extract_tree_nodes, plot_factor_tree,
    extract_factor_fingerprint, extract_fingerprint_matrix,
    compute_stimulus_similarity, project_stimuli_onto_tree,
    extract_factor_tree_nodes, compute_factor_activations,
    nodes_at_layer,
    select_class_circuit, per_class_accuracy,
    plot_factor_overview_panel, plot_factor_gallery, plot_input_layer_factors,
    plot_pruning_results, plot_embedding_comparison,
    compute_nmf_stability, plot_nmf_stability_figure,
    compute_k_sensitivity, plot_k_sensitivity_figure,
    plot_robustness_summary,
)

plt.rcParams.update({'figure.dpi': 80})
DEVICE = ('cuda' if torch.cuda.is_available() else
          'mps'  if torch.backends.mps.is_available() else 'cpu')
print('device:', DEVICE)

device: cuda


In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
IMAGENET_DIR = '../data'

CACHE_ROOT = '../data/cache/nb05_imagenet_squeezenet'
FIG_DIR    = '../figs/05_imagenet_squeezenet'
os.makedirs(CACHE_ROOT, exist_ok=True)
os.makedirs(FIG_DIR,    exist_ok=True)

# ── ImageNet constants ────────────────────────────────────────────────────────
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
N_CLASSES     = 1000
IMG_SIZE      = 224

# ── Focus categories ──────────────────────────────────────────────────────────
N_SAMPLES_PER_CATEGORY = 50

CATEGORY_CLASSES = {
    'airplane': [404, 895, 908],
    'ship':     [403, 472, 484],
    'car':      [407, 436, 468],
    'bicycle':  [444, 671],
    'elephant': [101, 385, 386],
    'bear':     [294, 295, 296],
    'dog':      [151, 152, 153],
    'bird':     [7, 8, 9],
}

CATEGORY_NAMES = list(CATEGORY_CLASSES.keys())
N_FOCUS        = len(CATEGORY_NAMES)
ALL_FOCUS_IDX  = sorted(set(i for idxs in CATEGORY_CLASSES.values() for i in idxs))
IDX_TO_CAT     = {idx: ci for ci, idxs in enumerate(CATEGORY_CLASSES.values()) for idx in idxs}

# Dict class_names for plot utilities
CAT_CLASS_NAMES = {i: CATEGORY_NAMES[i] for i in range(N_FOCUS)}

# ── BFT hyperparameters ───────────────────────────────────────────────────────
K_MAX      = [4, 4, 4, 4, 4, 6, 6, 6, 6, N_FOCUS]
N_BRANCHES = [1, 1, 1, 1, 1, 1, 1, 1, 2, 5]
POOL_METHOD    = 'avg'
STIM_THRESHOLD = 0.0

# ── Ablation ──────────────────────────────────────────────────────────────────
ABLATION_FRACS   = [0.02, 0.05, 0.10, 0.20, 0.30]
N_RANDOM_REPEATS = 3
USE_CACHED_ABL   = True

METHODS       = ['bft_top', 'bft_bottom', 'magnitude', 'random']
METHOD_COLORS = {'bft_top': 'red', 'bft_bottom': 'orange',
                 'magnitude': 'gray', 'random': 'black'}
METHOD_LABELS = {'bft_top': 'BFT most important', 'bft_bottom': 'BFT least important',
                 'magnitude': 'Highest magnitude', 'random': 'Random'}

print('Config ready.')
print(f'Categories ({N_FOCUS}): {CATEGORY_NAMES}')
print(f'Total focus ImageNet classes: {len(ALL_FOCUS_IDX)}')

Config ready.
Categories (8): ['airplane', 'ship', 'car', 'bicycle', 'elephant', 'bear', 'dog', 'bird']
Total focus ImageNet classes: 23


## §1 — Model

**SqueezeNet 1.1** pretrained on ImageNet-1K. Downloads ~4.7 MB on first run.

In [3]:
# ── Data loaders ──────────────────────────────────────────────────────────────
normalize = T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
test_tfm  = T.Compose([
    T.Resize(256),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    normalize,
])

def _make_imagenet(split, transform):
    """Load ImageNet; falls back to ImageFolder if ILSVRC structure absent."""
    try:
        return torchvision.datasets.ImageNet(IMAGENET_DIR, split=split, transform=transform)
    except Exception:
        folder = 'train' if split == 'train' else 'val'
        return torchvision.datasets.ImageFolder(
            os.path.join(IMAGENET_DIR, folder), transform=transform)

val_ds   = _make_imagenet('val', test_tfm)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

CLASS_NAMES = val_ds.classes  # synset IDs, e.g. 'n01440764'
print(f'Val: {len(val_ds):,}   Classes: {N_CLASSES}')
print(f'Focus class synsets: {[CLASS_NAMES[c] for c in ALL_FOCUS_IDX]}')

Val: 50,000   Classes: 1000
Focus class synsets: ['n01514668', 'n01514859', 'n01518878', 'n01871265', 'n02085620', 'n02085782', 'n02085936', 'n02132136', 'n02133161', 'n02134084', 'n02504013', 'n02504458', 'n02687172', 'n02690373', 'n02701002', 'n02814533', 'n02835271', 'n02930766', 'n02951358', 'n02981792', 'n03792782', 'n04552348', 'n04592741']


/home/jb3879/Factor_Trace/.venv/lib64/python3.11/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 3, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [4]:
# ── Focused val loader (focus categories only) ────────────────────────────────
val_targets  = np.array(val_ds.targets)
focus_idx    = np.where(np.isin(val_targets, ALL_FOCUS_IDX))[0]
focus_val_ds = Subset(val_ds, focus_idx)
focus_loader = DataLoader(focus_val_ds, batch_size=256, shuffle=False, num_workers=4)
print(f'Focus val samples: {len(focus_val_ds)}')

Focus val samples: 1150


In [5]:
# ── Load pretrained SqueezeNet 1.1 ────────────────────────────────────────────
model = squeezenet1_1(weights=SqueezeNet1_1_Weights.IMAGENET1K_V1).to(DEVICE)
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'SqueezeNet 1.1 — parameters: {n_params:,}')

# Per-category top-1 accuracy on focus val samples
cat_correct = np.zeros(N_FOCUS)
cat_total   = np.zeros(N_FOCUS)
with torch.no_grad():
    for x, y in focus_loader:
        x = x.to(DEVICE)
        preds    = model(x).argmax(1).cpu().numpy()
        yt       = y.numpy()
        true_cats = np.array([IDX_TO_CAT.get(int(t), -1) for t in yt])
        pred_cats = np.array([IDX_TO_CAT.get(int(p), -1) for p in preds])
        for ci in range(N_FOCUS):
            mask = true_cats == ci
            cat_correct[ci] += (pred_cats[mask] == ci).sum()
            cat_total[ci]   += mask.sum()

cat_acc = cat_correct / (cat_total + 1e-12)
for ci, name in enumerate(CATEGORY_NAMES):
    print(f'  {name:10s}  Top-1: {cat_acc[ci]:.3f}  ({int(cat_correct[ci])}/{int(cat_total[ci])})')
print(f'\nMean category Top-1: {cat_acc.mean():.3f}')
print('(Full val set: Top-1 ≈ 0.582, Top-5 ≈ 0.806 per torchvision)')

SqueezeNet 1.1 — parameters: 1,235,496


/home/jb3879/Factor_Trace/.venv/lib64/python3.11/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 3, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  airplane    Top-1: 0.847  (127/150)
  ship        Top-1: 0.687  (103/150)
  car         Top-1: 0.673  (101/150)
  bicycle     Top-1: 0.650  (65/100)
  elephant    Top-1: 0.833  (125/150)
  bear        Top-1: 0.747  (112/150)
  dog         Top-1: 0.567  (85/150)
  bird        Top-1: 0.833  (125/150)

Mean category Top-1: 0.730
(Full val set: Top-1 ≈ 0.582, Top-5 ≈ 0.806 per torchvision)


## §2 — BFT Factorization & Inspection

Layer data is collected via the **squeeze spine filter** — only the initial conv,
the 8 squeeze convs inside each Fire module, and the final classifier conv are captured.
This preserves BFT's sequential-layer assumption despite SqueezeNet's parallel expand branches.

In [6]:
def imdenorm(img, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    m = np.array(mean)[:, None, None]; s = np.array(std)[:, None, None]
    return np.clip((img * s + m).transpose(1, 2, 0), 0, 1)

def squeezenet_spine_filter(name, mod):
    """Select the sequential squeeze-spine: initial conv, squeeze convs, classifier conv."""
    return (name == 'features.0' or
            name == 'classifier.1' or
            (isinstance(mod, nn.Conv2d) and name.endswith('.squeeze')))

def filter_by_category(raw, n_per_category,
                        idx_to_cat=IDX_TO_CAT, n_categories=N_FOCUS):
    """Map ImageNet indices → category labels (0–N_FOCUS-1), sample n_per_category each.

    Samples are sorted by model confidence (highest first) within each category.
    Returns (filtered_dict, keep_indices).
    """
    orig_targets = raw['targets']
    cat_targets  = np.array([idx_to_cat.get(int(t), -1) for t in orig_targets])
    keep = []
    for ci in range(n_categories):
        ci_idx = np.where(cat_targets == ci)[0]
        if 'confidences' in raw and len(ci_idx):
            ci_idx = ci_idx[np.argsort(raw['confidences'][ci_idx])[::-1]]
        if len(ci_idx) > n_per_category:
            ci_idx = ci_idx[:n_per_category]
        keep.append(ci_idx)
    keep = np.sort(np.concatenate(keep))
    layer_data = [{**ld, 'input_fmap':  ld['input_fmap'][keep],
                         'output_fmap': ld['output_fmap'][keep]}
                  for ld in raw['layer_data']]
    result = {'images':       raw['images'][keep],
              'targets':      cat_targets[keep],      # category labels 0–7
              'orig_targets': orig_targets[keep],     # original ImageNet indices
              'layer_data':   layer_data}
    if 'confidences' in raw:
        result['confidences'] = raw['confidences'][keep]
    return result, keep

_p = os.path.join(CACHE_ROOT, 'raw_data_focus.pkl')
if os.path.exists(_p):
    with open(_p, 'rb') as f:
        raw0 = pickle.load(f)
    print('Loaded raw layer data from cache')
else:
    print('Collecting focus-category val data (squeeze spine only) …')
    raw0 = collect_layer_dicts(model, focus_loader, DEVICE,
                                only_correct=True,
                                layer_filter=squeezenet_spine_filter)
    with open(_p, 'wb') as f:
        pickle.dump(raw0, f)
    print('Saved.')

data0, _ = filter_by_category(raw0, N_SAMPLES_PER_CATEGORY)
all_images0   = data0['images']
all_targets0  = data0['targets']      # category labels 0–7
layer_inputs0 = [ld['input_fmap'] for ld in data0['layer_data']]
n_samples0    = len(all_images0)

print(f'{n_samples0} samples | {len(layer_inputs0)} spine layers')
print(f'Layer shapes: {[x.shape for x in layer_inputs0]}')
print(f'Layer names: {[ld["name"] for ld in data0["layer_data"]]}')
print(f'Category counts: {dict(zip(CATEGORY_NAMES, [int((all_targets0==i).sum()) for i in range(N_FOCUS)]))}')


Saved.
400 samples | 10 spine layers
Layer shapes: [(400, 3, 224, 224), (400, 64, 55, 55), (400, 128, 55, 55), (400, 128, 27, 27), (400, 256, 27, 27), (400, 256, 13, 13), (400, 384, 13, 13), (400, 384, 13, 13), (400, 512, 13, 13), (400, 512, 13, 13)]
Layer names: ['features.0', 'features.3.squeeze', 'features.4.squeeze', 'features.6.squeeze', 'features.7.squeeze', 'features.9.squeeze', 'features.10.squeeze', 'features.11.squeeze', 'features.12.squeeze', 'classifier.1']
Category counts: {'airplane': 50, 'ship': 50, 'car': 50, 'bicycle': 50, 'elephant': 50, 'bear': 50, 'dog': 50, 'bird': 50}


In [7]:
# ── Run BFT ───────────────────────────────────────────────────────────────────

print('Running BFT …')
tree_root0 = bft(
    data0['layer_data'],
    k_max=K_MAX, n_branches=N_BRANCHES,
    conv_pool_method=POOL_METHOD,
    stimulus_threshold=STIM_THRESHOLD,
    weighting='img_selectivity', verbose=1, n_jobs=3,
)

tree_nodes0   = extract_tree_nodes(tree_root0)
factor_nodes0 = extract_factor_tree_nodes(tree_root0)
l0_nodes0     = nodes_at_layer(tree_root0, 0)
K_root        = len(tree_root0['lambdas'])
print(f'Tree nodes: {len(tree_nodes0)}   K_root: {K_root}   L0 leaves: {len(l0_nodes0)}')

Running BFT …
[BFT] Layer 10/10 'classifier.1' (conv)  path=[]


/home/jb3879/Factor_Trace/.venv/lib64/python3.11/site-packages/sklearn/decomposition/_nmf.py:2306: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/home/jb3879/Factor_Trace/.venv/lib64/python3.11/site-packages/sklearn/decomposition/_nmf.py:2306: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


[BFT L10 'classifier.1' (conv)]   K=8  t=587.328s
[BFT] Layer 9/10 'features.12.squeeze' (conv)  path=[0]


/home/jb3879/Factor_Trace/.venv/lib64/python3.11/site-packages/sklearn/decomposition/_nmf.py:2306: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/home/jb3879/Factor_Trace/.venv/lib64/python3.11/site-packages/sklearn/decomposition/_nmf.py:2306: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


[BFT L9 'features.12.squeeze' (conv)]   K=6  t=100.403s
[BFT] Layer 8/10 'features.11.squeeze' (conv)  path=[0, 0]


/home/jb3879/Factor_Trace/.venv/lib64/python3.11/site-packages/sklearn/decomposition/_nmf.py:2306: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/home/jb3879/Factor_Trace/.venv/lib64/python3.11/site-packages/sklearn/decomposition/_nmf.py:2306: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


[BFT L8 'features.11.squeeze' (conv)]   K=6  t=102.372s
[BFT] Layer 7/10 'features.10.squeeze' (conv)  path=[0, 0, 0]


/home/jb3879/Factor_Trace/.venv/lib64/python3.11/site-packages/sklearn/decomposition/_nmf.py:2306: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
# ── Plot 1+4: Factor overview panels & per-factor galleries (all tree nodes) ──
for node in tree_nodes0:
    layer_name = node.get('layer_name', f'L{node["layer_idx"]}')
    figs1 = plot_factor_overview_panel(node, all_images0, all_targets0, CAT_CLASS_NAMES)
    for k, fig in enumerate(figs1):
        fig.savefig(os.path.join(FIG_DIR, f'factor_overview_{layer_name}_k{k}.pdf'),
                    bbox_inches='tight')
        plt.close(fig)
    K = node['img_factors'].shape[1]
    for k in range(K):
        fig4 = plot_factor_gallery(node, all_images0, all_targets0, CAT_CLASS_NAMES, k=k, n=10)
        fig4.savefig(os.path.join(FIG_DIR, f'factor_gallery_{layer_name}_k{k}.pdf'),
                     bbox_inches='tight')
        plt.close(fig4)

print(f'Plot 1+4 saved for {len(tree_nodes0)} tree nodes.')

In [ ]:
# ── Plot 2: Input-layer spatial factors (conv_rgb for initial conv, bars for squeeze) ──
for leaf in l0_nodes0:
    layer_name = leaf.get('layer_name', f'L{leaf["layer_idx"]}')
    ld = data0['layer_data'][leaf['layer_idx']]
    C_in = ld['weight'].shape[1]
    arch = 'conv_rgb' if C_in == 3 else 'fc'  # initial conv is RGB; squeeze convs are 1×1
    figs2 = plot_input_layer_factors(leaf, all_images0, arch=arch,
                                      image_shape=(3, IMG_SIZE, IMG_SIZE))
    for k, fig in enumerate(figs2):
        fig.savefig(os.path.join(FIG_DIR, f'input_factors_{layer_name}_k{k}.pdf'),
                    bbox_inches='tight')
        plt.close(fig)

print(f'Plot 2 saved for {len(l0_nodes0)} L0 leaf nodes.')

In [ ]:
# ── Spatial activation maps (only for initial conv — it has spatial extent) ───
def get_spatial_activation_map(model, images_np, node, layer_data, device,
                                n_images=6, which='top'):
    scores  = node['img_factors'][:, 0]
    sel_idx = np.argsort(scores)[::-1][:n_images] if which == 'top' else np.argsort(scores)[:n_images]
    ld      = layer_data[node['layer_idx']]
    C_out, C_in, kH, kW = ld['weight'].shape
    neu_f  = node['neural_factors']
    ch_imp = np.maximum(neu_f[:, 0].reshape(C_out, C_in * kH * kW).sum(1), 0)
    if ch_imp.sum() > 0: ch_imp /= ch_imp.sum()

    fmaps = {}
    tmod  = dict(model.named_modules())[node['layer_name']]
    hook  = tmod.register_forward_hook(lambda m, i, o: fmaps.update({'out': o.detach().cpu()}))
    model.eval()
    with torch.no_grad():
        model(torch.from_numpy(images_np[sel_idx]).float().to(device))
    hook.remove()
    fmap    = fmaps['out'].numpy()
    spatial = np.maximum((fmap * ch_imp[None, :, None, None]).sum(1), 0)
    return spatial, sel_idx

# Only the initial conv (features.0) has spatial kernels; squeeze convs are 1×1
spatial_leaves = [n for n in l0_nodes0 if n['layer_name'] == 'features.0']
for leaf in spatial_leaves[:2]:
    top_maps, top_idx = get_spatial_activation_map(model, all_images0, leaf,
                                                    data0['layer_data'], DEVICE, 6, 'top')
    bot_maps, bot_idx = get_spatial_activation_map(model, all_images0, leaf,
                                                    data0['layer_data'], DEVICE, 6, 'bottom')
    n = len(top_idx)
    fig, axes = plt.subplots(4, n, figsize=(2.2 * n, 8))
    for col in range(n):
        axes[0, col].imshow(imdenorm(all_images0[top_idx[col]]))
        axes[0, col].set_title(CLASS_NAMES[all_targets0[top_idx[col]]][:8], fontsize=8)
        axes[0, col].axis('off')
        axes[1, col].imshow(top_maps[col], cmap='hot'); axes[1, col].axis('off')
        axes[2, col].imshow(imdenorm(all_images0[bot_idx[col]]))
        axes[2, col].set_title(CLASS_NAMES[all_targets0[bot_idx[col]]][:8], fontsize=8)
        axes[2, col].axis('off')
        axes[3, col].imshow(bot_maps[col], cmap='hot'); axes[3, col].axis('off')
    for row, lbl in enumerate(['Top image', 'Top map', 'Bot image', 'Bot map']):
        axes[row, 0].set_ylabel(lbl)
    fig.suptitle(f'{leaf["layer_name"]} path={leaf["path"]}: spatial activation maps')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f'spatial_{leaf["layer_name"]}.pdf'), bbox_inches='tight')
    plt.show()

In [ ]:
# ── Scaffold graph ────────────────────────────────────────────────────────────
# all_targets0 already contains category labels 0–(N_FOCUS-1)
scaffold_edges   = build_scaffold_edges(tree_root0)
scaffold_loading = scaffold_loading_from_edges(scaffold_edges, all_targets0, N_FOCUS)
fig, ax = plt.subplots(figsize=(14, 6))
plot_scaffold_graph(scaffold_edges, scaffold_loading, ax=ax)
ax.set_title('Scaffold graph — SqueezeNet 1.1 (node colour = dominant focus category)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'scaffold.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 5: Robustness analysis ───────────────────────────────────────────────
from src.bft import compute_conv_joint_arbors as _conv_arb
from sklearn.metrics.pairwise import cosine_similarity as _cos_sim_nb
from scipy.optimize import linear_sum_assignment

# Root-level joint arbor matrix (needed for 5a, 5c, 5d)
root_ld    = data0['layer_data'][tree_root0['layer_idx']]
X_root_raw = _conv_arb(root_ld['weight'], root_ld['input_fmap'],
                        pool_method=POOL_METHOD)
X_root = np.clip(X_root_raw, 0, None)
K_r    = int(tree_root0['img_factors'].shape[1])
print(f'Root X_joint: {X_root.shape}   K_r={K_r}')

# ── 5a: NMF initialisation robustness ────────────────────────────────────────
print('5a: NMF init robustness (10 seeds) ...')
sim_5a, factors_5a = compute_nmf_stability(X_root, K_r, n_seeds=10)
fig_5a = plot_nmf_stability_figure(sim_5a,
                                    title=f'Root: NMF init robustness (K={K_r}, 10 seeds)')
fig_5a.savefig(os.path.join(FIG_DIR, 'robustness_5a_nmf_init.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig_5a)

# ── 5b: Model-seed robustness — SKIPPED ──────────────────────────────────────
print('5b: Skipped — single pretrained SqueezeNet 1.1 (no multi-seed training).')

# ── 5c: K-sensitivity ─────────────────────────────────────────────────────────
print('5c: K-sensitivity (K*±1, 5 seeds) ...')
k_sens_result, _ = compute_k_sensitivity(X_root, k_star=K_r, n_seeds=5)
fig_5c = plot_k_sensitivity_figure(k_sens_result, k_star=K_r,
                                    title=f'Root: K-sensitivity (K*={K_r})')
fig_5c.savefig(os.path.join(FIG_DIR, 'robustness_5c_k_sensitivity.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig_5c)

# ── 5d: Dataset-split robustness (val set, 50/50) ─────────────────────────────
print('5d: Dataset split robustness (50/50 val splits) ...')
rng_split = np.random.default_rng(42)
perm_s    = rng_split.permutation(n_samples0)
split_A   = perm_s[:n_samples0 // 2]
split_B   = perm_s[n_samples0 // 2:]

X_A = np.clip(_conv_arb(root_ld['weight'], root_ld['input_fmap'][split_A],
                          pool_method=POOL_METHOD), 0, None)
X_B = np.clip(_conv_arb(root_ld['weight'], root_ld['input_fmap'][split_B],
                          pool_method=POOL_METHOD), 0, None)

_, factors_A = compute_nmf_stability(X_A, K_r, n_seeds=5)
_, factors_B = compute_nmf_stability(X_B, K_r, n_seeds=5)

# Cross-split similarity: align seed-0 factor from A to seed-0 from B
H_A = factors_A[0]; H_B = factors_B[0]   # (n_feat, K_r) normalised
S_cross = _cos_sim_nb(H_A.T, H_B.T)      # (K_r, K_r)
row_ind, col_ind = linear_sum_assignment(-S_cross)
cross_sim = np.array([S_cross[i, j] for i, j in zip(row_ind, col_ind)])

fig_5d, ax5d = plt.subplots(figsize=(4, 4))
ax5d.boxplot(cross_sim, widths=0.45,
             medianprops=dict(color='black', linewidth=1.5))
ax5d.axhline(0.9, ls='--', color='#e15759', lw=1.5, label='threshold 0.90')
ax5d.set_ylabel('cosine similarity (split A vs B)')
ax5d.set_xticklabels([f'K={K_r} factors'])
ax5d.set_ylim(0, 1.05)
ax5d.legend(fontsize=8)
ax5d.set_title('Dataset split robustness\n(50/50 val splits, root node)', fontsize=9)
fig_5d.tight_layout()
fig_5d.savefig(os.path.join(FIG_DIR, 'robustness_5d_split.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig_5d)
print('Plot 5 complete.')

## §3 — Ablation (focus classes)

BFT vs magnitude vs random ablation sweep over the squeeze spine layers.
Weight lookup uses `ld['name'] + '.weight'` (robust for any layer filter).

In [ ]:
# ── Ablation helpers ──────────────────────────────────────────────────────────
def cnn_score_magnitude(layer_data):
    """Magnitude scores {(layer_idx, i, j): |W[i,j]|}."""
    scores = {}
    for l_idx, ld in enumerate(layer_data):
        W = ld['weight']
        flat = np.abs(W.reshape(W.shape[0], -1))
        for i in range(flat.shape[0]):
            for j in range(flat.shape[1]):
                scores[(l_idx, i, j)] = float(flat[i, j])
    return scores

def cnn_ablate_model(model, layer_data, scores, frac, method='top', seed=42):
    """Zero `frac` fraction of weights by score. method in {'top','bottom','random'}."""
    all_vals = np.array(list(scores.values()))
    keys = list(scores.keys())
    n_ablate = max(1, int(round(len(keys) * frac)))
    if method == 'random':
        rng = np.random.RandomState(seed)
        ablate_keys = set(map(tuple, np.array(keys)[rng.permutation(len(keys))[:n_ablate]]))
    elif method == 'top':
        threshold = np.sort(all_vals)[::-1][n_ablate - 1]
        ablate_keys = {k for k, v in scores.items() if v >= threshold}
    else:
        threshold = np.sort(all_vals)[n_ablate - 1]
        ablate_keys = {k for k, v in scores.items() if v <= threshold}

    abl   = copy.deepcopy(model)
    state = abl.state_dict()
    for l_idx, ld in enumerate(layer_data):
        pname = ld['name'] + '.weight'
        if pname not in state: continue
        W      = state[pname].clone()
        W_flat = W.reshape(W.shape[0], -1)
        for (li, i, j) in ablate_keys:
            if li == l_idx and i < W_flat.shape[0] and j < W_flat.shape[1]:
                W_flat[i, j] = 0.0
        state[pname] = W_flat.reshape(W.shape)
    abl.load_state_dict(state)
    return abl

def eval_focus_accuracy(model, loader, device,
                        idx_to_cat=IDX_TO_CAT, n_focus=N_FOCUS):
    """Per-category top-1; correct if predicted ImageNet class falls in same category."""
    model.eval()
    correct_arr = np.zeros(n_focus); total_arr = np.zeros(n_focus)
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            preds     = model(x).argmax(1).cpu().numpy()
            yt        = y.numpy()
            true_cats = np.array([idx_to_cat.get(int(t), -1) for t in yt])
            pred_cats = np.array([idx_to_cat.get(int(p), -1) for p in preds])
            for ci in range(n_focus):
                mask = true_cats == ci
                correct_arr[ci] += (pred_cats[mask] == ci).sum()
                total_arr[ci]   += mask.sum()
    return correct_arr / (total_arr + 1e-12)

print('Ablation helpers defined.')

In [ ]:
# ── Plot 6: Comparative ablation sweep (single pretrained seed) ───────────────
ABL_CACHE = os.path.join(CACHE_ROOT, 'plot6_ablation.pkl')

if USE_CACHED_ABL and os.path.exists(ABL_CACHE):
    with open(ABL_CACHE, 'rb') as f:
        pruning_raw = pickle.load(f)
    print('Loaded Plot 6 ablation from cache.')
else:
    mag_scores = cnn_score_magnitude(data0['layer_data'])
    pruning_raw = [{}]   # single pretrained seed
    for ci in range(N_FOCUS):
        cat_name = CATEGORY_NAMES[ci]
        bft_scores, info = select_class_circuit(tree_root0, all_targets0, ci)
        if bft_scores is None:
            bft_scores = mag_scores
        pruning_raw[0][ci] = {}
        print(f'  {cat_name}', end=' ', flush=True)
        for method in METHODS:
            if method == 'bft_top':
                scores_m, direction = bft_scores, 'top'
            elif method == 'bft_bottom':
                scores_m, direction = bft_scores, 'bottom'
            elif method == 'magnitude':
                scores_m, direction = mag_scores, 'top'
            else:  # random
                scores_m, direction = mag_scores, 'random'
            pruning_raw[0][ci][method] = {}
            for frac in ABLATION_FRACS:
                n_reps = N_RANDOM_REPEATS if method == 'random' else 1
                rep_accs = []
                for rep in range(n_reps):
                    abl = cnn_ablate_model(model, data0['layer_data'], scores_m, frac,
                                           method=direction, seed=rep)
                    accs_arr = eval_focus_accuracy(abl, focus_loader, DEVICE)
                    rep_accs.append({c: float(accs_arr[c]) for c in range(N_FOCUS)})
                pruning_raw[0][ci][method][frac] = {
                    c: float(np.mean([r[c] for r in rep_accs])) for c in range(N_FOCUS)
                }
        print('done')

    with open(ABL_CACHE, 'wb') as f:
        pickle.dump(pruning_raw, f)
    print('Plot 6 ablation done and cached.')

# Aggregate: single seed → stds are zero
pruning_data = {}
pruning_stds = {}
for ci in range(N_FOCUS):
    pruning_data[ci] = {}
    pruning_stds[ci] = {}
    for method in METHODS:
        pruning_data[ci][method] = {}
        pruning_stds[ci][method] = {}
        for frac in ABLATION_FRACS:
            pruning_data[ci][method][frac] = pruning_raw[0][ci][method][frac]
            pruning_stds[ci][method][frac] = {c: 0.0 for c in range(N_FOCUS)}

print(f'pruning_data ready: {N_FOCUS} categories × {len(METHODS)} methods × {len(ABLATION_FRACS)} fracs')

In [ ]:
# ── Plot 6 display ────────────────────────────────────────────────────────────
# Prepend frac=0 baseline (unpruned model)
fracs_plot        = [0.0] + ABLATION_FRACS
baseline_accs_arr = eval_focus_accuracy(model, focus_loader, DEVICE)
baseline_acc      = {c: float(baseline_accs_arr[c]) for c in range(N_FOCUS)}

for ci in range(N_FOCUS):
    for method in METHODS:
        pruning_data[ci][method][0.0] = baseline_acc
        pruning_stds[ci][method][0.0] = {c: 0.0 for c in range(N_FOCUS)}

figs6 = plot_pruning_results(
    pruning_data, CAT_CLASS_NAMES, METHODS, fracs_plot,
    method_colors=METHOD_COLORS, method_labels=METHOD_LABELS,
    pruning_stds=pruning_stds,
)
for ci, fig in enumerate(figs6):
    cat_name = CATEGORY_NAMES[ci]
    fig.savefig(os.path.join(FIG_DIR, f'pruning_{cat_name}.pdf'), bbox_inches='tight')
    plt.show()
    plt.close(fig)

print(f'Plot 6 saved ({len(figs6)} figures).')

## §4 — Stimulus Analysis via NNLS

Round-trip consistency, ID val-split sanity check, and far-OOD analysis.

In [ ]:
# ── Round-trip test ───────────────────────────────────────────────────────────
N_RT   = min(200, n_samples0)
rng_rt = np.random.default_rng(0)
rt_sub = rng_rt.choice(n_samples0, N_RT, replace=False)
rt_inputs = [l[rt_sub] for l in layer_inputs0]

projected_rt = project_stimuli_onto_tree(tree_root0, rt_inputs)
F_orig_rt    = extract_fingerprint_matrix(tree_root0, rt_sub)
F_rt         = extract_fingerprint_matrix(projected_rt, np.arange(N_RT))
rt_sims      = 1.0 - paired_cosine_distances(F_orig_rt, F_rt)
rt_root      = 1.0 - paired_cosine_distances(
    tree_root0['img_factors'][rt_sub], projected_rt['img_factors'])

print(f'Round-trip (full):  mean={rt_sims.mean():.4f}  std={rt_sims.std():.4f}')
print(f'Round-trip (root):  mean={rt_root.mean():.4f}  std={rt_root.std():.4f}')
print()
print('Active-sample fractions per node (first 8):')
for tn in tree_nodes0[:8]:
    sw = tn['stimulus_weights_in']
    print(f'  layer={tn["layer_idx"]}  path={tn["path"]}  active={(sw > 0.01).mean():.3f}')

In [ ]:
# ── ID sanity check: val split-A vs split-B cross-similarity (4 focus categories) ─
# Randomly partition each category's val samples into two halves (A / B).
# Half-A uses the original tree projection; half-B is re-projected via NNLS.
# Strong within-category / weak between-category similarity validates the BFT encoding.
rng_id   = np.random.default_rng(42)
SHOW_CI  = list(range(4))   # first 4 categories: airplane, ship, car, bicycle
N_BLK    = 40
blocks   = {}

for ci in SHOW_CI:
    ci_idx = np.where(all_targets0 == ci)[0].copy()
    rng_id.shuffle(ci_idx)
    half  = len(ci_idx) // 2
    idx_A = ci_idx[:half]
    idx_B = ci_idx[half:]
    if len(idx_A) > N_BLK: idx_A = rng_id.choice(idx_A, N_BLK, replace=False)
    if len(idx_B) > N_BLK: idx_B = rng_id.choice(idx_B, N_BLK, replace=False)
    li_B   = [l[idx_B] for l in layer_inputs0]
    proj_B = project_stimuli_onto_tree(tree_root0, li_B)
    if len(idx_A):
        blocks[f'A-{CATEGORY_NAMES[ci]}'] = extract_fingerprint_matrix(tree_root0, idx_A)
    if len(idx_B):
        blocks[f'B-{CATEGORY_NAMES[ci]}'] = extract_fingerprint_matrix(proj_B, np.arange(len(idx_B)))

F_cross  = np.concatenate(list(blocks.values()), axis=0)
bl_sizes = [len(v) for v in blocks.values()]
bl_ends  = list(np.cumsum(bl_sizes))
S_cross  = compute_stimulus_similarity(F_cross)
centres  = np.array([0] + bl_ends[:-1]) + np.array(bl_sizes) / 2

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(S_cross, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine similarity')
for b in bl_ends[:-1]:
    ax.axhline(b - 0.5, color='k', lw=1.5); ax.axvline(b - 0.5, color='k', lw=1.5)
ax.set_xticks(centres); ax.set_xticklabels(list(blocks), rotation=45, ha='right', fontsize=8)
ax.set_yticks(centres); ax.set_yticklabels(list(blocks), fontsize=8)
ax.set_title('ID Sanity Check: val split-A vs split-B cross-similarity (4 focus categories)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'id_cross_similarity.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Far-OOD: 4 synthetic 3-channel image types ────────────────────────────────
N_FAR = 200; rng_f = np.random.default_rng(99); C = 3
_mn   = np.array(IMAGENET_MEAN)[:, None, None]
_st   = np.array(IMAGENET_STD)[:, None, None]
_chk  = (np.indices((IMG_SIZE, IMG_SIZE)).sum(0) % 2)[None].astype(np.float32)

_noise_raw = np.clip(rng_f.normal(0.5, 0.25,
                     (N_FAR, C, IMG_SIZE, IMG_SIZE)).astype(np.float32), 0, 1)
_orig_px   = all_images0[:N_FAR] * _st + _mn
_inv_norm  = (np.clip(1.0 - _orig_px, 0, 1) - _mn) / _st

far_ood_arrays = {
    'gaussian_noise': ((_noise_raw - _mn) / _st).astype(np.float32),
    'uniform_gray':   np.zeros((N_FAR, C, IMG_SIZE, IMG_SIZE), dtype=np.float32),
    'checkerboard':   np.broadcast_to(_chk, (N_FAR, C, IMG_SIZE, IMG_SIZE)).copy().astype(np.float32),
    'inverted_test':  _inv_norm.astype(np.float32),
}

far_ood_data = {}
for name, imgs in far_ood_arrays.items():
    ds = TensorDataset(torch.from_numpy(imgs), torch.zeros(len(imgs), dtype=torch.long))
    d  = collect_layer_dicts(model, DataLoader(ds, 128, shuffle=False), DEVICE,
                              only_correct=False,
                              layer_filter=squeezenet_spine_filter)
    li = [ld['input_fmap'] for ld in d['layer_data']]
    d['layer_inputs']   = li
    d['projected_root'] = project_stimuli_onto_tree(tree_root0, li)
    d['factor_nodes']   = extract_factor_tree_nodes(d['projected_root'])
    far_ood_data[name]  = d
    print(f'{name:20s}  n={len(imgs)}')

n_ex  = 6
fig, axes = plt.subplots(len(far_ood_arrays), n_ex,
                          figsize=(n_ex * 2, len(far_ood_arrays) * 2.2))
for row, (name, imgs) in enumerate(far_ood_arrays.items()):
    for col in range(n_ex):
        raw = np.clip(imgs[col] * _st + _mn, 0, 1).transpose(1, 2, 0)
        axes[row, col].imshow(raw); axes[row, col].axis('off')
    axes[row, 0].set_ylabel(name, fontsize=9, rotation=30, ha='right', va='center')
plt.suptitle('Far OOD — example images', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'far_ood_examples.pdf'), bbox_inches='tight')
plt.show()

n_types = len(far_ood_data)
fig, axes = plt.subplots(1, n_types, figsize=(6 * n_types, 4.5))
for ax, (name, d) in zip(axes, far_ood_data.items()):
    all_idx = np.arange(len(d['images']))
    acts    = compute_factor_activations(d['factor_nodes'], all_idx)
    plot_factor_tree(d['factor_nodes'], acts, ax=ax, title=name)
plt.suptitle('Far OOD — factor tree activations', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'far_ood_trees.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 7: Embedding comparison (MDS fingerprints | PCA fingerprints | PCA last-layer) ──
N_EACH  = 60
rng_m   = np.random.default_rng(7)

F_parts, la_parts, emb_labels, emb_conditions = [], [], [], []

# ID: focus val categories
id_sub = rng_m.choice(n_samples0, min(N_EACH, n_samples0), replace=False)
F_parts.append(extract_fingerprint_matrix(tree_root0, id_sub))
la_parts.append(layer_inputs0[-1][id_sub].reshape(len(id_sub), -1))
emb_labels    += list(all_targets0[id_sub])
emb_conditions += ['ID'] * len(id_sub)

# Far-OOD
ood_names = list(far_ood_data.keys())
for oi, (name, d) in enumerate(far_ood_data.items()):
    sub = rng_m.choice(len(d['images']), min(N_EACH, len(d['images'])), replace=False)
    F_parts.append(extract_fingerprint_matrix(d['projected_root'], sub))
    la_parts.append(d['layer_inputs'][-1][sub].reshape(len(sub), -1))
    emb_labels    += [N_FOCUS + oi] * len(sub)
    emb_conditions += [name] * len(sub)

F_joint    = np.concatenate(F_parts,  axis=0)
la_joint   = np.concatenate(la_parts, axis=0)
emb_labels = np.array(emb_labels)

# Build full class_names dict: 0–7 = focus categories, 8+ = OOD condition names
full_class_names = dict(CAT_CLASS_NAMES)
for oi, name in enumerate(ood_names):
    full_class_names[N_FOCUS + oi] = name

fig7 = plot_embedding_comparison(
    F_joint, la_joint, emb_labels, full_class_names,
    condition_labels=emb_conditions,
    title='ID ImageNet focus classes vs Far-OOD',
)
fig7.savefig(os.path.join(FIG_DIR, 'embedding_comparison.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig7)

# Fingerprint intra vs inter-class similarity (ID focus classes only)
F_all = extract_fingerprint_matrix(tree_root0, np.arange(n_samples0))
S_all = compute_stimulus_similarity(F_all)
intra_vals, inter_vals = [], []
for fi in range(N_FOCUS):
    mask = all_targets0 == fi
    intra = S_all[np.ix_(mask, mask)]
    intra_vals.extend(intra[np.triu_indices_from(intra, k=1)])
    for fj in range(fi + 1, N_FOCUS):
        inter_vals.extend(S_all[np.ix_(mask, all_targets0 == fj)].ravel())

intra_arr = np.array(intra_vals); inter_arr = np.array(inter_vals)
print(f'Intra-class similarity: {intra_arr.mean():.3f} ± {intra_arr.std():.3f}')
print(f'Inter-class similarity: {inter_arr.mean():.3f} ± {inter_arr.std():.3f}')

fig_hist, ax_hist = plt.subplots(figsize=(6, 4))
ax_hist.hist(intra_arr, bins=60, alpha=0.6, label='Intra-class', density=True)
ax_hist.hist(inter_arr, bins=60, alpha=0.6, label='Inter-class', density=True)
ax_hist.axvline(intra_arr.mean(), color='C0', ls='--')
ax_hist.axvline(inter_arr.mean(), color='C1', ls='--')
ax_hist.set(xlabel='Cosine similarity', ylabel='Density',
            title='Factor fingerprint: intra vs inter-class similarity (focus classes)')
ax_hist.legend()
fig_hist.tight_layout()
fig_hist.savefig(os.path.join(FIG_DIR, 'fingerprint_intra_inter.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig_hist)